# EXACT API Tunnel Test Notebook

Notebook này chỉ để hiểu và test API format.

Nó tạo một FastAPI mini server ở `/predict`, sau đó mở public URL bằng Cloudflare Tunnel hoặc ngrok.

API format bắt buộc:
- Request: JSON object có `query_id`, `type`, `query`, `premises`, `options`.
- Response: JSON list, mỗi phần tử có `query_id`, `answer`, `unit`, `explanation`, `premises_used`, `reasoning`.


In [1]:
# CELL 0 — Install dependencies

!pip install -q fastapi uvicorn requests httptools


In [2]:
# CELL 1 — Create a minimal /predict API server

from pathlib import Path
import textwrap, os, subprocess, time, requests, json

PROJECT = Path("/kaggle/working/exact_api_test")
PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT / "app").mkdir(exist_ok=True)
(PROJECT / "app/__init__.py").write_text("")

(PROJECT / "app/main.py").write_text(textwrap.dedent(r"""
from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import Literal, Any
import re

app = FastAPI(title="EXACT API Test")

class PredictRequest(BaseModel):
    query_id: str
    type: Literal["type1", "type2"]
    query: str
    premises: list[str] = Field(default_factory=list)
    options: list[str] = Field(default_factory=list)

def choose_option(ans, options):
    # Portal Type1 usually uses Yes/No/Uncertain.
    # Our internal model may say Unknown, so map Unknown -> Uncertain if needed.
    if ans == "Unknown" and "Uncertain" in options:
        ans = "Uncertain"
    if options and ans not in options:
        if "Uncertain" in options:
            return "Uncertain"
        return options[0]
    return ans

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(req: PredictRequest):
    # IMPORTANT: response must be JSON LIST, not a single object.

    if req.type == "type2":
        q = req.query.lower()
        nums = re.findall(r"r\d*\s*=\s*([0-9.]+)\s*ohm", q)
        v = re.search(r"([0-9.]+)\s*v\b", q)

        if "parallel" in q and "current" in q and len(nums) >= 2 and v:
            rs = [float(x) for x in nums[:2]]
            V = float(v.group(1))
            I = sum(V / r for r in rs)
            ans = str(int(round(I))) if abs(I - round(I)) < 1e-9 else f"{I:.6g}"
            return [{
                "query_id": req.query_id,
                "answer": ans,
                "unit": "A",
                "explanation": "Computed by deterministic formula solver.",
                "premises_used": [],
                "reasoning": {
                    "type": "cot",
                    "steps": [
                        "Compute branch currents with I=U/R.",
                        "Add branch currents.",
                        f"Result: {ans} A."
                    ]
                }
            }]

        return [{
            "query_id": req.query_id,
            "answer": "0",
            "unit": "",
            "explanation": "The Type2 fallback could not solve this confidently.",
            "premises_used": [],
            "reasoning": None
        }]

    # Minimal Type1 demo logic.
    # In the real notebook, replace this with LoRA + parser + v35 verifier.
    q = req.query.lower()
    premises = " ".join(req.premises).lower()

    if "receive a scholarship" in q and "every student receives a scholarship" in premises:
        ans = "Yes"
        reasoning = {"type": "proof", "steps": ["Universal premise establishes existence in this simplified smoke case."]}
        premises_used = [0]
    elif "publish a paper" in q and "receives a scholarship" in premises:
        ans = "Unknown"
        reasoning = None
        premises_used = []
    elif "pass the exam" in q and "no student passes the exam" in premises:
        ans = "No"
        reasoning = {"type": "proof", "steps": ["No-student premise refutes existence."]}
        premises_used = [0]
    else:
        ans = "Unknown"
        reasoning = None
        premises_used = []

    ans = choose_option(ans, req.options)

    return [{
        "query_id": req.query_id,
        "answer": ans,
        "unit": "",
        "explanation": "Derived by the Type1 API smoke-test logic.",
        "premises_used": premises_used,
        "reasoning": reasoning
    }]
""").strip() + "\n")

print("Created:", PROJECT)
print((PROJECT / "app/main.py").read_text()[:2000])


Created: /kaggle/working/exact_api_test
from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import Literal, Any
import re

app = FastAPI(title="EXACT API Test")

class PredictRequest(BaseModel):
    query_id: str
    type: Literal["type1", "type2"]
    query: str
    premises: list[str] = Field(default_factory=list)
    options: list[str] = Field(default_factory=list)

def choose_option(ans, options):
    # Portal Type1 usually uses Yes/No/Uncertain.
    # Our internal model may say Unknown, so map Unknown -> Uncertain if needed.
    if ans == "Unknown" and "Uncertain" in options:
        ans = "Uncertain"
    if options and ans not in options:
        if "Uncertain" in options:
            return "Uncertain"
        return options[0]
    return ans

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(req: PredictRequest):
    # IMPORTANT: response must be JSON LIST, not a single object.

    if req.type == "type2":

In [3]:
# CELL 2 — Start local FastAPI server at port 9000

import os, subprocess, time, requests, pathlib

subprocess.run("pkill -f uvicorn || true", shell=True)
time.sleep(2)

log_path = "/kaggle/working/exact_api_test.log"
log = open(log_path, "w")

env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT)

p_api = subprocess.Popen(
    ["python", "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "9000"],
    cwd=str(PROJECT),
    stdout=log,
    stderr=log,
    env=env,
)

time.sleep(3)
print("API pid:", p_api.pid)
print("health:", requests.get("http://127.0.0.1:9000/health", timeout=5).text)
print(pathlib.Path(log_path).read_text()[-2000:])


API pid: 105
health: {"status":"ok"}
INFO:     Started server process [105]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9000 (Press CTRL+C to quit)
INFO:     127.0.0.1:37646 - "GET /health HTTP/1.1" 200 OK



In [4]:
# CELL 3 — Test local /predict API format

import requests, json, time

LOCAL_PREDICT_URL = "http://127.0.0.1:9000/predict"

tests = [
    {
        "query_id": "T1_yes",
        "type": "type1",
        "query": "Does at least one student receive a scholarship?",
        "premises": ["Every student receives a scholarship."],
        "options": ["Yes", "No", "Uncertain"],
    },
    {
        "query_id": "T1_unknown",
        "type": "type1",
        "query": "Does at least one student publish a paper?",
        "premises": ["Every student receives a scholarship."],
        "options": ["Yes", "No", "Uncertain"],
    },
    {
        "query_id": "T2_parallel",
        "type": "type2",
        "query": "Two resistors R1 = 4 ohm and R2 = 6 ohm are in parallel across a 12 V battery. Find the total current.",
        "premises": [],
        "options": [],
    },
]

for payload in tests:
    r = requests.post(LOCAL_PREDICT_URL, json=payload, timeout=30)
    print("\nREQUEST:")
    print(json.dumps(payload, ensure_ascii=False, indent=2))
    print("STATUS:", r.status_code)
    print("RESPONSE:")
    print(json.dumps(r.json(), ensure_ascii=False, indent=2))



REQUEST:
{
  "query_id": "T1_yes",
  "type": "type1",
  "query": "Does at least one student receive a scholarship?",
  "premises": [
    "Every student receives a scholarship."
  ],
  "options": [
    "Yes",
    "No",
    "Uncertain"
  ]
}
STATUS: 200
RESPONSE:
[
  {
    "query_id": "T1_yes",
    "answer": "Yes",
    "unit": "",
    "explanation": "Derived by the Type1 API smoke-test logic.",
    "premises_used": [
      0
    ],
    "reasoning": {
      "type": "proof",
      "steps": [
        "Universal premise establishes existence in this simplified smoke case."
      ]
    }
  }
]

REQUEST:
{
  "query_id": "T1_unknown",
  "type": "type1",
  "query": "Does at least one student publish a paper?",
  "premises": [
    "Every student receives a scholarship."
  ],
  "options": [
    "Yes",
    "No",
    "Uncertain"
  ]
}
STATUS: 200
RESPONSE:
[
  {
    "query_id": "T1_unknown",
    "answer": "Uncertain",
    "unit": "",
    "explanation": "Derived by the Type1 API smoke-test logic.",


In [5]:
# CELL 4A — Open public URL with Cloudflare Tunnel

# This usually does not require an account.
# The URL will look like https://xxxxx.trycloudflare.com/predict

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared

import subprocess, time, pathlib, re

cf_log = "/kaggle/working/cloudflared.log"

p_cf = subprocess.Popen(
    ["/kaggle/working/cloudflared", "tunnel", "--url", "http://127.0.0.1:9000", "--no-autoupdate"],
    stdout=open(cf_log, "w"),
    stderr=subprocess.STDOUT,
)

time.sleep(12)
text = pathlib.Path(cf_log).read_text()
print(text[-5000:])

m = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", text)
if not m:
    raise RuntimeError("Could not find trycloudflare URL in log. Re-run this cell or inspect /kaggle/working/cloudflared.log")

PUBLIC_BASE_URL = m.group(0)
PUBLIC_PREDICT_URL = PUBLIC_BASE_URL + "/predict"

print("PUBLIC_BASE_URL:", PUBLIC_BASE_URL)
print("PUBLIC_PREDICT_URL:", PUBLIC_PREDICT_URL)


--------------------------------------------------------------------------------------------+
2026-06-13T17:43:27Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-13T17:43:27Z INF |  https://dollars-near-wildlife-though.trycloudflare.com                                    |
2026-06-13T17:43:27Z INF +--------------------------------------------------------------------------------------------+
2026-06-13T17:43:27Z INF Cannot determine default configuration path. No file [config.yml config.yaml] in [~/.cloudflared ~/.cloudflare-warp ~/cloudflare-warp /etc/cloudflared /usr/local/etc/cloudflared]
2026-06-13T17:43:27Z INF Version 2026.6.0 (Checksum 08d27c4c5d3ed73ee3e98ef2ddceb4ad09fd4cfc28e243565a189538e8ccd706)
2026-06-13T17:43:27Z INF GOOS: linux, GOVersion: go1.26.4, GoArch: amd64
2026-06-13T17:43:27Z INF Settings: map[ha-connections:1 no-autoupdate:true protocol:quic url:http://127.0.0.1:9000]
2026-06-13T17:43:27Z INF Generated 

In [ ]:
# CELL 4B — Alternative public URL with ngrok

# Use this only if Cloudflare Tunnel fails.
# If ngrok asks for auth token, get it from https://dashboard.ngrok.com/get-started/your-authtoken

# !pip install -q pyngrok
# from pyngrok import ngrok
# ngrok.set_auth_token("PASTE_YOUR_NGROK_TOKEN_HERE")
# public_url = ngrok.connect(9000, "http")
# PUBLIC_BASE_URL = str(public_url)
# PUBLIC_PREDICT_URL = PUBLIC_BASE_URL + "/predict"
# print("PUBLIC_PREDICT_URL:", PUBLIC_PREDICT_URL)


In [7]:
import requests, json

url = "https://dollars-near-wildlife-though.trycloudflare.com/predict"

payload = {
    "query_id": "T2_public_test",
    "type": "type2",
    "query": "Two resistors R1 = 4 ohm and R2 = 6 ohm are in parallel across a 12 V battery. Find the total current.",
    "premises": [],
    "options": []
}

r = requests.post(url, json=payload, timeout=60)
print(r.status_code)
print(r.text)

200
[{"query_id":"T2_public_test","answer":"5","unit":"A","explanation":"Computed by deterministic formula solver.","premises_used":[],"reasoning":{"type":"cot","steps":["Compute branch currents with I=U/R.","Add branch currents.","Result: 5 A."]}}]


In [8]:
# CELL 6 — API format cheat sheet

print("""
Portal wants this URL:
  https://your-public-domain/predict

Request payload example:
{
  "query_id": "abc",
  "type": "type1",
  "query": "...",
  "premises": ["...", "..."],
  "options": ["Yes", "No", "Uncertain"]
}

Response must be a JSON LIST:
[
  {
    "query_id": "abc",
    "answer": "Yes",
    "unit": "",
    "explanation": "Must be non-empty.",
    "premises_used": [0],
    "reasoning": null
  }
]

For Type2:
- answer should be numeric string, e.g. "5"
- unit should be correct ASCII unit, e.g. "A", "ohm", "V"
""")



Portal wants this URL:
  https://your-public-domain/predict

Request payload example:
{
  "query_id": "abc",
  "type": "type1",
  "query": "...",
  "premises": ["...", "..."],
  "options": ["Yes", "No", "Uncertain"]
}

Response must be a JSON LIST:
[
  {
    "query_id": "abc",
    "answer": "Yes",
    "unit": "",
    "explanation": "Must be non-empty.",
    "premises_used": [0],
    "reasoning": null
  }
]

For Type2:
- answer should be numeric string, e.g. "5"
- unit should be correct ASCII unit, e.g. "A", "ohm", "V"



In [ ]:
# CELL 7 — Stop local server/tunnel when finished

# Run this only after submission/testing is done.
# import subprocess
# subprocess.run("pkill -f uvicorn || true", shell=True)
# subprocess.run("pkill -f cloudflared || true", shell=True)
# print("Stopped.")
